# Служебная тетрадь: координаты районов Москвы

**Что делает.** Собирает координаты центров 120 районов из датасета и сохраняет их в `../imports/district_coords.csv` (столбцы `District, lat, lon, source`).

**Зачем.** Из координат основная тетрадь строит признаки «расстояние до центра» и «направление от центра». Так модель узнаёт, какие районы находятся рядом, а категория района этого не говорит.

**Источник.** Страницы районов в английской Википедии, координаты из метаданных страницы (`wgCoordinates`). У 5 районов на странице нет координат. Для них центр района взят по карте вручную, в столбце `source` они помечены как `manual`.

**Запуск.** Нужен интернет, сбор идёт ~2–3 минуты. Основная тетрадь эту тетрадь не запускает, а читает готовый CSV.

In [1]:
import re
import time
import urllib.parse
import urllib.request

import pandas as pd

OUT = "../imports/district_coords.csv"

Соответствие названий района в датасете и страниц английской Википедии. Названия транслитерированы, часть порядков слов отличается (например, «Дегунино Западное» → `Zapadnoye_Degunino_District`).

In [2]:
WIKI_TITLES = {
    "Академический": "Akademichesky_District",
    "Алексеевский": None,
    "Алтуфьевский": "Altufyevsky_District",
    "Арбат": "Arbat_District",
    "Аэропорт": "Aeroport_District",
    "Бабушкинский": None,
    "Басманный": "Basmanny_District",
    "Беговой": "Begovoy_District",
    "Бескудниковский": "Beskudnikovsky_District",
    "Бибирево": "Bibirevo_District",
    "Бирюлёво Восточное": "Biryulyovo_Vostochnoye_District",
    "Бирюлёво Западное": "Biryulyovo_Zapadnoye_District",
    "Богородский": "Bogorodskoye_District",
    "Братеево": "Brateyevo_District",
    "Бутырский": "Butyrsky_District",
    "Вешняки": "Veshnyaki_District",
    "Внуково": "Vnukovo_District",
    "Войковский": "Voykovsky_District",
    "Восточный": "Vostochny_District,_Moscow",
    "Гагаринский": "Gagarinsky_District,_Moscow",
    "Головинский": "Golovinsky_District",
    "Гольяново": None,
    "Даниловский": "Danilovsky_District,_Moscow",
    "Дегунино Восточное": "Vostochnoye_Degunino_District",
    "Дегунино Западное": "Zapadnoye_Degunino_District",
    "Дмитровский": "Dmitrovsky_District,_Moscow",
    "Донской": "Donskoy_District",
    "Дорогомилово": "Dorogomilovo_District",
    "Жулебино": "Vykhino-Zhulebino_District",
    "Замоскворечье": "Zamoskvorechye_District",
    "Зюзино": "Zyuzino_District",
    "Зябликово": "Zyablikovo_District",
    "Ивановское": "Ivanovskoye_District",
    "Измайлово": "Izmaylovo_District",
    "Измайлово Восточное": "Vostochnoye_Izmaylovo_District",
    "Капотня": "Kapotnya_District",
    "Коньково": "Konkovo_District",
    "Коптево": "Koptevo_District",
    "Косино-Ухтомский": "Kosino-Ukhtomsky_District",
    "Котловка": "Kotlovka_District",
    "Красносельский": "Krasnoselsky_District,_Moscow",
    "Крылатский": "Krylatskoye_District",
    "Кузьминки": "Kuzminki_District",
    "Кунцево": "Kuntsevo_District",
    "Куркино": "Kurkino_District",
    "Левобережный": "Levoberezhny_District,_Moscow",
    "Лефортово": "Lefortovo_District",
    "Лианозово": "Lianozovo_District",
    "Ломоносовский": "Lomonosovsky_District,_Moscow",
    "Лосиноостровский": "Losinoostrovsky_District",
    "Люблино": "Lyublino_District",
    "Марфино": "Marfino_District",
    "Марьина роща": "Maryina_Roshcha_District",
    "Марьино": "Maryino_District",
    "Медведково Южное": "Yuzhnoye_Medvedkovo_District",
    "Метрогородок": "Metrogorodok_District",
    "Мещанский": "Meshchansky_District",
    "Митино": "Mitino_District",
    "Можайский": "Mozhaysky_District,_Moscow",
    "Молжаниновский": "Molzhaninovsky_District",
    "Москворечье-Сабурово": "Moskvorechye-Saburovo_District",
    "Нагатино Садовники": "Nagatino-Sadovniki_District",
    "Нагатинский затон": "Nagatinsky_Zaton_District",
    "Нагорный": "Nagorny_District",
    "Некрасовка": "Nekrasovka_District",
    "Нижегородский": "Nizhegorodsky_District,_Moscow",
    "Ново-Переделкино": "Novo-Peredelkino_District",
    "Новогиреево": "Novogireyevo_District",
    "Новокосино": "Novokosino_District",
    "Обручевский": "Obruchevsky_District",
    "Останкино": "Ostankinsky_District",
    "Отрадное": "Otradnoye_District",
    "Очаково-Матвеевское": "Ochakovo-Matveyevskoye_District",
    "Перово": "Perovo_District",
    "Печатники": "Pechatniki_District",
    "Покровское Стрешнево": "Pokrovskoye-Streshnevo_District",
    "Преображенский": "Preobrazhenskoye_District",
    "Пресненский": None,
    "Район Вернадского": "Prospekt_Vernadskogo_District",
    "Раменки": "Ramenki_District",
    "Ростокино": "Rostokino_District",
    "Рязанский": "Ryazansky_District,_Moscow",
    "Савеловский": "Savyolovsky_District",
    "Свиблово": "Sviblovo_District",
    "Северное Бутово": "Severnoye_Butovo_District",
    "Северное Измайлово": "Severnoye_Izmaylovo_District",
    "Северное Медведково": "Severnoye_Medvedkovo_District",
    "Северное Орехово Борисово": "Orekhovo-Borisovo_Severnoye_District",
    "Северное Тушино": "Severnoye_Tushino_District",
    "Северное Чертаново": "Chertanovo_Severnoye_District",
    "Северный": "Severny_District,_Moscow",
    "Сокол": "Sokol_District",
    "Соколиная гора": "Sokolinaya_Gora_District",
    "Сокольники": "Sokolniki_District",
    "Солнцево": None,
    "Строгино": "Strogino_District",
    "Таганка": "Tagansky_District",
    "Тверской": "Tverskoy_District",
    "Текстильщики": "Tekstilshchiki_District",
    "Тимирязевский": "Timiryazevsky_District",
    "Тропарево Никулино": "Troparyovo-Nikulino_District",
    "Тёплый Стан": "Tyoply_Stan_District",
    "Филевский парк": "Filyovsky_Park_District",
    "Фили Давыдково": "Fili-Davydkovo_District",
    "Хамовники": "Khamovniki_District",
    "Ховрино": "Khovrino_District",
    "Хорошевский": "Khoroshyovsky_District",
    "Хорошёво-Мневники": "Khoroshyovo-Mnyovniki_District",
    "Царицыно": "Tsaritsyno_District",
    "Центральное Чертаново": "Chertanovo_Tsentralnoye_District",
    "Черемушки": "Cheryomushki_District",
    "Щукино": "Shchukino_District",
    "Южное Бутово": "Yuzhnoye_Butovo_District",
    "Южное Орехово Борисово": "Orekhovo-Borisovo_Yuzhnoye_District",
    "Южное Тушино": "Yuzhnoye_Tushino_District",
    "Южное Чертаново": "Chertanovo_Yuzhnoye_District",
    "Южнопортовый": "Yuzhnoportovy_District",
    "Якиманка": "Yakimanka_District",
    "Ярославский": "Yaroslavsky_District,_Moscow",
    "Ясенево": "Yasenevo_District"
}

Районы, у страниц которых нет координат: центр района по карте.

In [3]:
MANUAL = {  # страницы без координат: центр района по карте
    "Алексеевский": (55.807, 37.640),
    "Бабушкинский": (55.868, 37.664),
    "Гольяново": (55.823, 37.818),
    "Пресненский": (55.758, 37.560),
    "Солнцево": (55.649, 37.395),
}

Загрузка страницы и извлечение координат. При ошибке сети — до трёх повторов с паузой.

In [4]:
def fetch_coords(title):
    url = "https://en.wikipedia.org/wiki/" + urllib.parse.quote(title)
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-pro-study/1.0"})
    for attempt in range(3):
        try:
            html = urllib.request.urlopen(req, timeout=30).read().decode("utf-8")
            c = re.search(r'"wgCoordinates":\{"lat":([\d.]+),"lon":([\d.]+)\}', html)
            return (float(c.group(1)), float(c.group(2))) if c else None
        except Exception:
            time.sleep(5 * (attempt + 1))
    return None

In [5]:
rows = []
for district, title in WIKI_TITLES.items():
    if district in MANUAL:
        rows.append((district, *MANUAL[district], "manual"))
        continue
    ll = fetch_coords(title)
    rows.append((district, *(ll or (None, None)), "wikipedia" if ll else "missing"))
    time.sleep(0.7)  # бережём Википедию от частых запросов
coords = pd.DataFrame(rows, columns=["District", "lat", "lon", "source"])
coords.source.value_counts()

source
wikipedia    115
manual         5
Name: count, dtype: int64

Проверка: все точки внутри Москвы, пропусков нет. Сохраняем.

In [6]:
assert coords.lat.between(55.4, 56.1).all() and coords.lon.between(37.1, 38.1).all()
coords.to_csv(OUT, index=False)
coords.head()

,District,lat,lon,source
0,Академический,55.689722,37.573056,wikipedia
1,Алексеевский,55.807000,37.640000,manual
2,Алтуфьевский,55.879722,37.584167,wikipedia
3,Арбат,55.751000,37.590000,wikipedia
4,Аэропорт,55.791389,37.559444,wikipedia
